# Pipeline RAW → Tensore Normalizzato
### Sezione 6.2 — Implementazione completa con `rawpy.postprocess()`

Questo notebook implementa e visualizza ogni passo della pipeline descritta nella
**Sezione 6.2**, trasformando un file RAW grezzo in un tensore normalizzato pronto
per una rete neurale.

Alcuni passaggi (demosaicatura AHD, white balance, color matrix, Bradford) sono
delegati a **rawpy/libraw** per correttezza numerica e compatibilità con migliaia
di modelli di fotocamera. Dove la pipeline Python differisce dalla formulazione
teorica, la differenza è evidenziata esplicitamente.

---

### Mappa teoria → implementazione

| Sezione 6.2 | Operazione | Dove avviene |
|-------------|-----------|--------------|
| 6.2.1 | Linearizzazione $R_{norm}$ | `rawpy` internamente + visualizzazione manuale |
| 6.2.2 | Modello rumore Poisson-Gaussiano | Visualizzazione analitica (non applicata) |
| 6.2.3 | Demosaicatura AHD | `rawpy.postprocess()` |
| 6.2.4 | Correzione vignettatura + CA | `rawpy.postprocess()` (profili DNG) |
| 6.2.5 | White balance | `rawpy.postprocess(use_camera_wb=True)` |
| 6.2.6 | Camera RGB → XYZ (color matrix) | `rawpy.postprocess(output_color=sRGB)` |
| 6.2.7 | XYZ → sRGB lineare (Bradford) | `rawpy.postprocess(output_color=sRGB)` |
| 6.2.8 | Gamma encoding sRGB | **Implementazione manuale** |
| 6.2.9 | Downscaling Lanczos | **Implementazione manuale** |
| 6.2.10 | Normalizzazione ImageNet | **Implementazione manuale** |

---
## Setup e dipendenze

```
pip install rawpy Pillow numpy matplotlib scipy
```

In [ ]:
import rawpy
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from scipy import fft as scipy_fft
from scipy.stats import norm as sp_norm
import os, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor':   '#1a1a1a',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#888',
    'ytick.color':      '#888',
    'text.color':       '#eee',
    'figure.titlesize': 14,
    'axes.titlesize':   11,
    'font.family':      'monospace',
})

RAW_PATH        = '../images/DSC05370.ARW'   # ARW / DNG / NEF / CR3 / RAF …
TRAIN_LONG_SIDE = 768                         # L_train per il downscaling

assert os.path.exists(RAW_PATH), f'File non trovato: {RAW_PATH}'
print(f'✓  {RAW_PATH}   ({os.path.getsize(RAW_PATH)/1e6:.1f} MB)')

---
## Step 0 — Lettura metadati e mosaico CFA grezzo
### Sezione 6.2.1 (parziale)

Prima di qualsiasi elaborazione estraiamo i parametri del sensore dall'header
del file RAW. Questi parametri sono usati sia da rawpy internamente, sia nelle
nostre visualizzazioni esplicite.

### 6.2.1 — Linearizzazione del segnale RAW

Il sensore introduce due artefatti sistematici: il **livello di nero** (dark current,
offset fisso dell'amplificatore) e il **livello di saturazione** (full-well capacity).

Sia $\mathbf{R} \in \mathbb{Z}^{H_{raw}\times W_{raw}}$ la lettura grezza del sensore
con bit depth $b$ (tipicamente 14 bit per Sony ARW → valori in $[0, 16383]$).

La **normalizzazione lineare** è:

$$R_{norm}(i,j) = \text{clip}\!\left(\frac{\mathbf{R}(i,j) - b_{dark}}{b_{sat} - b_{dark}},\; 0,\; 1\right) \in [0,1]$$

dove $b_{dark}$ (livello di nero, tipicamente 512–1024) e $b_{sat}$ (livello di
saturazione, tipicamente $2^{14}-1 = 16383$) sono letti dall'header DNG/EXIF.

**Proprietà fondamentale:** il segnale normalizzato è **linearmente proporzionale
all'irradianza della scena**:
$$R_{norm}(i,j) \approx k \cdot E(i,j) \cdot t_{exp}$$

dove $k$ è la responsività del sensore (funzione dell'ISO) e $t_{exp}$ il tempo
di esposizione. Questa linearità distingue il RAW da JPEG: il RAW conserva la
piena gamma dinamica della scena.

> ⚠️ **Nota implementativa:** questa linearizzazione è eseguita internamente da
> `rawpy.postprocess()`. La calcoliamo esplicitamente **solo** per visualizzare il
> mosaico CFA grezzo nello Step 0. Il valore `R_norm` calcolato qui **non entra**
> nella pipeline di produzione.

### Pattern CFA — Sezione 6.2.3 (preambolo)

Il sensore produce un'immagine monocanale dove ogni pixel misura l'irradianza
attraverso un solo filtro cromatico. Per il pattern RGGB:

$$c(i,j) = \begin{cases}
R   & i \equiv 0 \pmod 2,\ j \equiv 0 \pmod 2 \\
G_1 & i \equiv 0 \pmod 2,\ j \equiv 1 \pmod 2 \\
G_2 & i \equiv 1 \pmod 2,\ j \equiv 0 \pmod 2 \\
B   & i \equiv 1 \pmod 2,\ j \equiv 1 \pmod 2
\end{cases}$$

La visualizzazione del crop 128×128 mostra il pattern Bayer: ogni pixel ha
un solo colore, gli altri due mancanti devono essere stimati dalla demosaicatura.

In [ ]:
raw = rawpy.imread(RAW_PATH)

# ── Parametri sensore (header DNG/EXIF) ──────────────────────────────────────
b_dark = float(np.mean(raw.black_level_per_channel))  # media su 4 canali RGGB
b_sat  = float(raw.white_level)

# ── White balance (usato da rawpy internamente, mostrato per trasparenza) ─────
wb_raw  = np.array(raw.camera_whitebalance[:3], dtype=np.float32)
wb_norm = wb_raw / wb_raw[1]   # normalizzato su G: w_G = 1

# ── Color matrix cam→XYZ D50 (usata da rawpy internamente) ───────────────────
# rawpy espone XYZ→cam; invertiamo con pseudo-inversa per ottenere cam→XYZ
cm = raw.color_matrix[:3, :3].astype(np.float64)
M_cam2xyz = np.linalg.pinv(cm) if not np.all(cm == 0) else np.eye(3)

# ── Pattern CFA ───────────────────────────────────────────────────────────────
pattern_map = {(0,1,3,2):'RGGB',(2,3,1,0):'BGGR',(1,0,2,3):'GRBG',(3,2,0,1):'GBRG'}
cfa_name = pattern_map.get(tuple(raw.raw_pattern.flatten()), 'custom')

# ── ISO (via rawpy, fallback exifread, fallback 800) ──────────────────────────
iso_value = None
if hasattr(raw, 'camera_params'):
    iso_value = getattr(raw.camera_params, 'iso_speed', None) or None
if not iso_value:
    try:
        import exifread
        with open(RAW_PATH, 'rb') as f:
            tags = exifread.process_file(f, stop_tag='EXIF ISOSpeedRatings', details=False)
        tag = tags.get('EXIF ISOSpeedRatings')
        if tag: iso_value = int(str(tag))
    except ImportError:
        pass
iso_value = iso_value or 800

print(f'Dimensioni RAW:    {raw.raw_image.shape[0]} × {raw.raw_image.shape[1]} px')
print(f'b_dark:            {b_dark:.1f}')
print(f'b_sat:             {b_sat:.0f}  (range utile: {b_sat - b_dark:.0f} livelli)')
print(f'WB (norm):         wR={wb_norm[0]:.4f}  wG={wb_norm[1]:.4f}  wB={wb_norm[2]:.4f}')
print(f'Pattern CFA:       {cfa_name}  {raw.raw_pattern.tolist()}')
print(f'ISO:               {iso_value}')
print(f'M_cam→XYZ (D50):\n{np.round(M_cam2xyz, 4)}')

# ── Linearizzazione manuale — solo per visualizzazione ───────────────────────
raw_vis  = raw.raw_image_visible.astype(np.float32)
H_raw, W_raw = raw_vis.shape
R_norm   = np.clip((raw_vis - b_dark) / (b_sat - b_dark), 0.0, 1.0)
TH = 8
thumb_raw = R_norm[::TH, ::TH]

# ── Crop 128×128 centrato per mostrare il pattern Bayer ───────────────────────
cy, cx = H_raw//2, W_raw//2
bayer_crop = R_norm[cy:cy+128, cx:cx+128]
bayer_vis  = np.zeros((128, 128, 3), dtype=np.float32)
bayer_vis[0::2, 0::2, 0] = bayer_crop[0::2, 0::2]  # R
bayer_vis[0::2, 1::2, 1] = bayer_crop[0::2, 1::2]  # G1
bayer_vis[1::2, 0::2, 1] = bayer_crop[1::2, 0::2]  # G2
bayer_vis[1::2, 1::2, 2] = bayer_crop[1::2, 1::2]  # B

# ── Figura ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(thumb_raw ** (1/2.2), cmap='gray', interpolation='lanczos')
axes[0].set_title(f'RAW grezzo R_norm (mosaico {cfa_name})\nthumbnail 1:{TH}, gamma preview')
axes[0].axis('off')

axes[1].imshow(np.clip(bayer_vis, 0, 1) ** (1/2.2), interpolation='nearest')
axes[1].set_title('Pattern Bayer — crop centrale 128×128 px\n(R=rosso, G1/G2=verde, B=blu)')
axes[1].axis('off')

x_adc = np.linspace(0, b_sat + 200, 1000)
y_lin = np.clip((x_adc - b_dark) / (b_sat - b_dark), 0, 1)
axes[2].plot(x_adc, y_lin, color='#80cbc4', lw=2.5)
axes[2].axvline(b_dark, color='cyan',   lw=1.2, ls='--', label=f'b_dark = {b_dark:.0f}')
axes[2].axvline(b_sat,  color='yellow', lw=1.2, ls='--', label=f'b_sat  = {b_sat:.0f}')
axes[2].fill_between(x_adc, y_lin, alpha=0.1, color='#80cbc4')
axes[2].set_title('Curva di linearizzazione\n$R_{norm} = (R - b_{dark}) / (b_{sat} - b_{dark})$')
axes[2].set_xlabel('valore grezzo ADC')
axes[2].set_ylabel('$R_{norm}$')
axes[2].legend(fontsize=9)
axes[2].set_xlim(0, b_sat + 200)

fig.suptitle('Step 0 — Metadati sensore, linearizzazione e mosaico CFA', fontsize=13)
plt.tight_layout()
plt.show()

---
## Step 0b — Modello di rumore Poisson-Gaussiano
### Sezione 6.2.2

> ⚠️ **Differenza teoria/implementazione:** la Sezione 6.2.2 descrive una
> normalizzazione SNR-invariante $\tilde{R}_{norm}$ da applicare al segnale
> prima della demosaicatura. In questa pipeline, **non applichiamo questa
> normalizzazione esplicitamente**: rawpy gestisce internamente la coerenza
> tra ISO diversi attraverso i guadagni di amplificazione. Questa cella
> visualizza il modello di rumore analiticamente, a scopo didattico.

### Modello Poisson-Gaussiano eteroschedastico

Il segnale $R_{norm}$ è corrotto da rumore di due nature fisicamente distinte:

**Rumore di shot (Poisson):** originato dalla natura quantistica della luce.
Il numero di fotoni rilevati per pixel segue una distribuzione di Poisson,
la cui varianza è uguale al valor medio:
$$\text{Var}[N_{photon}] = \mathbb{E}[N_{photon}]$$

**Rumore di lettura (Gaussiano):** introdotto dall'amplificatore analogico e
dal convertitore A/D, indipendente dal segnale.

Il modello combinato per la varianza al pixel $(i,j)$ è:

$$\sigma^2_{noise}(i,j) = \underbrace{\alpha \cdot R_{norm}(i,j)}_{\text{shot noise}} + \underbrace{\sigma^2_{read}}_{\text{read noise}}$$

con:
- $\alpha$ = gain factor, funzione dell'ISO: $\alpha \approx \alpha_0 \cdot (\text{ISO}/\text{ISO}_0)$
- $\sigma^2_{read}$ = varianza di lettura, caratteristica del sensore (cresce con ISO)

La normalizzazione SNR-invariante teorica sarebbe:

$$\tilde{R}_{norm}(i,j) = \frac{R_{norm}(i,j)}{\sqrt{\sigma^2_{noise}(i,j)} + \varepsilon}$$

Questa è differenziabile rispetto a $R_{norm}$ e renderebbe la distribuzione
del segnale più uniforme tra immagini ad alto e basso ISO.

**Perché non la applichiamo:** rawpy con `use_camera_wb=True` e `no_auto_bright=True`
produce output già calibrato per i guadagni ISO. Applicare ulteriormente
$\tilde{R}_{norm}$ prima del downscaling introdurrebbe una normalizzazione
locale che altererebbe la gamma dinamica in modo non invertibile.

In [ ]:
# ── Parametri modello Poisson-Gaussiano (valori tipici per Sony A-series) ────
ISO_BASE        = 100
ALPHA_BASE      = 0.0008    # gain a ISO 100
SIGMA_READ_BASE = 0.004     # σ_read a ISO 100

alpha       = ALPHA_BASE * (iso_value / ISO_BASE)
sigma_read2 = (SIGMA_READ_BASE * (iso_value / ISO_BASE)) ** 2
eps         = 1e-6

print(f'ISO corrente:     {iso_value}')
print(f'α (gain factor):  {alpha:.6f}')
print(f'σ²_read:          {sigma_read2:.8f}')

# ── Mappa di varianza sul thumbnail ──────────────────────────────────────────
sigma2_map = alpha * thumb_raw + sigma_read2
snr_map    = thumb_raw / (np.sqrt(sigma2_map) + eps)

# ── Figure ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

axes[0,0].imshow(thumb_raw ** (1/2.2), cmap='gray', interpolation='lanczos')
axes[0,0].set_title('$R_{norm}$ — segnale linearizzato')
axes[0,0].axis('off')

im1 = axes[0,1].imshow(np.sqrt(sigma2_map), cmap='inferno', interpolation='lanczos')
axes[0,1].set_title(f'$\sigma_{{noise}}(i,j)$ — mappa deviazione std\n(ISO={iso_value})')
axes[0,1].axis('off')
plt.colorbar(im1, ax=axes[0,1], fraction=0.046)

im2 = axes[0,2].imshow(np.log1p(snr_map), cmap='viridis', interpolation='lanczos')
axes[0,2].set_title('$\log(1 + \text{SNR}(i,j))$\n(alto = segnale pulito)')
axes[0,2].axis('off')
plt.colorbar(im2, ax=axes[0,2], fraction=0.046)

# Curva σ²(s) vs segnale per vari ISO
signal_v = np.linspace(0, 1, 500)
iso_list  = [100, 400, 1600, 6400]
colors_iso = ['#4fc3f7','#81c784','#ffb74d','#e57373']
for iso_i, col in zip(iso_list, colors_iso):
    a_i = ALPHA_BASE * (iso_i / ISO_BASE)
    r_i = (SIGMA_READ_BASE * (iso_i / ISO_BASE)) ** 2
    axes[1,0].plot(signal_v, a_i * signal_v + r_i, color=col, lw=1.8, label=f'ISO {iso_i}')
axes[1,0].set_title('$\sigma^2_{noise} = \alpha s + \sigma^2_{read}$\nvs segnale, per vari ISO')
axes[1,0].set_xlabel('$R_{norm}$ (segnale)')
axes[1,0].set_ylabel('$\sigma^2_{noise}$')
axes[1,0].legend(fontsize=8)

# Decomposizione shot vs read a ISO corrente
shot_v = alpha * signal_v
read_v = np.full_like(signal_v, sigma_read2)
axes[1,1].stackplot(signal_v, shot_v, read_v,
                    labels=['shot noise (Poisson) $\alpha s$',
                            'read noise (Gaussiano) $\sigma^2_{read}$'],
                    colors=['#42a5f5','#ef5350'], alpha=0.7)
axes[1,1].set_title(f'Decomposizione rumore (ISO={iso_value})')
axes[1,1].set_xlabel('$R_{norm}$')
axes[1,1].set_ylabel('$\sigma^2$')
axes[1,1].legend(fontsize=8)

# SNR vs segnale per vari ISO
for iso_i, col in zip(iso_list, colors_iso):
    a_i = ALPHA_BASE * (iso_i / ISO_BASE)
    r_i = (SIGMA_READ_BASE * (iso_i / ISO_BASE)) ** 2
    snr_i = signal_v / (np.sqrt(a_i * signal_v + r_i) + eps)
    axes[1,2].plot(signal_v, snr_i, color=col, lw=1.8, label=f'ISO {iso_i}')
axes[1,2].set_title('SNR$(s)$ = $s\,/\,\sigma_{noise}(s)$\nvs segnale, per vari ISO')
axes[1,2].set_xlabel('$R_{norm}$')
axes[1,2].set_ylabel('SNR')
axes[1,2].legend(fontsize=8)

fig.suptitle('Step 0b — Modello rumore Poisson-Gaussiano (analisi, non applicato)', fontsize=13)
plt.tight_layout()
plt.show()

---
## Step 1 — `rawpy.postprocess()`: demosaicatura AHD, WB, color matrix
### Sezioni 6.2.3, 6.2.4, 6.2.5, 6.2.6, 6.2.7

Questo singolo passaggio implementa internamente **cinque sezioni** della pipeline
teorica. I parametri sono scelti con cura per ottenere dati **lineari puri** in
uscita (nessuna gamma applicata).

---

### 6.2.3 — Demosaicatura AHD

L'algoritmo **AHD** (Adaptive Homogeneity-Directed, Hirakawa & Parks 2005)
stima $\hat{\mathbf{E}}^{full} \in [0,1]^{H\times W\times 3}$ in tre fasi:

**Fase 1 — Interpolazione adattiva del verde:**
$G$ ha doppia densità rispetto a R e B (due campioni per blocco $2\times 2$)
ed è interpolato in direzione orizzontale $H$ e verticale $V$ separatamente:
$$\hat{G}^H(i,j) = \frac{R_{norm}(i,j-1) + R_{norm}(i,j+1)}{2} + \frac{2R_{norm}(i,j) - R_{norm}(i,j-2) - R_{norm}(i,j+2)}{4}$$

**Fase 2 — Differenze cromatiche:**
R e B sono interpolati tramite $\hat{D}_c = E^c - G$, più smooth spazialmente
degli assoluti (riduce gli artefatti ai bordi cromatici):
$$\hat{E}^{full}_R = \hat{D}_R + \hat{G}, \qquad \hat{E}^{full}_B = \hat{D}_B + \hat{G}$$

**Fase 3 — Selezione adattiva:**
Per ogni pixel si calcola l'omogeneità locale in ciascuna direzione e si sceglie
quella con minore variazione:
$$\hat{E}_c(i,j) = \begin{cases} \hat{E}^H_c & H(i,j) \leq V(i,j) \\ \hat{E}^V_c & \text{altrimenti} \end{cases}$$
dove $H(i,j) = \sum_{(i',j') \in \mathcal{N}(i,j)} |\Delta E^H(i',j')|$.

---

### 6.2.4 — Correzione ottica (vignettatura e aberrazione cromatica)

> ⚠️ **Differenza teoria/implementazione:** la Sezione 6.2.4 descrive la
> correzione esplicita di vignettatura ($v(r) = 1 + k_2r^2 + k_4r^4 + k_6r^6$)
> e aberrazione cromatica laterale ($\rho_c = (1 + \Delta r_c \cdot r^2)(i,j)$).
> rawpy applica queste correzioni **solo se il file RAW contiene i profili
> DNG opcodes** (principalmente DNG e alcuni RAW con profili Adobe incorporati).
> Per ARW senza profilo incorporato, questi artefatti **non vengono corretti**
> in questo step — sarebbero da applicare manualmente con i profili Adobe.

---

### 6.2.5 — White balance

La correzione dell'illuminante applica una moltiplicazione per canale con i
guadagni $\mathbf{w}_{wb} = (w_R, w_G, w_B)$ dai metadati EXIF:
$$\hat{E}^{wb}_c(i,j) = w_c \cdot \hat{E}^{full}_c(i,j), \quad w_G = 1 \text{ (convenzione)}$$

Fisicamente: se la scena è a luce tungsteno (\~3200K), il sensore sovra-risponde
al rosso → $w_R < 1$ (riduce), $w_B > 1$ (amplifica).

---

### 6.2.6 + 6.2.7 — Camera RGB → XYZ → sRGB lineare

rawpy applica in sequenza la **color matrix** e l'**adattamento di Bradford**:

$$\mathbf{XYZ}_{D50} = \mathbf{M}_{cam\to XYZ} \cdot \hat{\mathbf{E}}^{wb}$$

$$\mathbf{XYZ}_{D65} = \mathbf{M}_{Bradford} \cdot \mathbf{XYZ}_{D50}, \quad
\mathbf{M}_{Bradford} = \begin{pmatrix} 0.9556 & -0.0230 & 0.0632 \\ -0.0283 & 1.0099 & 0.0210 \\ 0.0123 & -0.0205 & 1.3299 \end{pmatrix}$$

$$\mathbf{I}^{lin} = \mathbf{M}_{XYZ\to sRGB} \cdot \mathbf{XYZ}_{D65}, \quad
\mathbf{M}_{XYZ\to sRGB} = \begin{pmatrix} 3.2406 & -1.5372 & -0.4986 \\ -0.9689 & 1.8758 & 0.0415 \\ 0.0557 & -0.2040 & 1.0570 \end{pmatrix}$$

Output: $\mathbf{I}^{lin} \in [0,1]^{H\times W\times 3}$ in spazio **sRGB lineare**.

---

### Parametri critici di `postprocess()`

| Parametro | Valore scelto | Motivazione |
|-----------|--------------|-------------|
| `demosaic_algorithm` | `AHD` | Qualità fotografica, adattivo ai bordi (Sezione 6.2.3) |
| `output_color` | `sRGB` | Esegue cam→XYZ(D50)→XYZ(D65)→sRGB (Sezioni 6.2.6–7) |
| `output_bps` | `16` | Massima precisione numerica (65535 livelli vs 255) |
| `use_camera_wb` | `True` | Usa guadagni WB misurati dalla camera (Sezione 6.2.5) |
| `no_auto_bright` | `True` | **Disabilita** l'auto-exposure: dati proporzionali all'irradianza |
| `gamma` | `(1, 1)` | **Nessuna gamma applicata**: output $I^{lin}$ lineare puro |
| `highlight_mode` | `Clip` | Clip pulito a 1.0 per pixel saturi (no ricostruzione highlight) |

> ⚠️ **Il parametro più critico è `gamma=(1,1)`:** senza di esso rawpy
> applicherebbe la curva sRGB internamente (equivalente alla Sezione 6.2.8),
> rendendo impossibile distinguere il dominio lineare nel passaggio successivo.
> Con `(1,1)` l'output è puramente lineare: $I^{lin} \propto E(i,j)$.

In [ ]:
print('Esecuzione rawpy.postprocess() ...')
rgb16 = raw.postprocess(
    demosaic_algorithm = rawpy.DemosaicAlgorithm.AHD,   # Sezione 6.2.3
    output_color       = rawpy.ColorSpace.sRGB,          # Sezioni 6.2.6–6.2.7
    output_bps         = 16,                             # 65535 livelli
    use_camera_wb      = True,                           # Sezione 6.2.5
    no_auto_bright     = True,                           # no auto-exposure
    gamma              = (1, 1),                         # ← nessuna gamma!
    highlight_mode     = rawpy.HighlightMode.Clip,
)

# uint16 → float32 [0,1]
I_lin = rgb16.astype(np.float32) / 65535.0
H, W  = I_lin.shape[:2]

print(f'I_lin shape: {I_lin.shape}  dtype={I_lin.dtype}')
print(f'  R: [{I_lin[:,:,0].min():.4f}, {I_lin[:,:,0].max():.4f}]  mean={I_lin[:,:,0].mean():.4f}')
print(f'  G: [{I_lin[:,:,1].min():.4f}, {I_lin[:,:,1].max():.4f}]  mean={I_lin[:,:,1].mean():.4f}')
print(f'  B: [{I_lin[:,:,2].min():.4f}, {I_lin[:,:,2].max():.4f}]  mean={I_lin[:,:,2].mean():.4f}')

# Luminanza CIE Y (coefficienti standard sRGB primaries)
# Y = 0.2126 R + 0.7152 G + 0.0722 B
lum = 0.2126*I_lin[:,:,0] + 0.7152*I_lin[:,:,1] + 0.0722*I_lin[:,:,2]

TH = 8
thumb_lin = I_lin[::TH, ::TH]

fig, axes = plt.subplots(2, 5, figsize=(22, 8))

# ── Riga 0: immagini ──────────────────────────────────────────────────────────
axes[0,0].imshow(thumb_raw ** (1/2.2), cmap='gray', interpolation='lanczos')
axes[0,0].set_title('RAW grezzo $R_{norm}$\n(mosaico CFA, pre-demosaicatura)')
axes[0,0].axis('off')

axes[0,1].imshow(np.clip(thumb_lin, 0, 1) ** (1/2.2), interpolation='lanczos')
axes[0,1].set_title('$I^{lin}$ — sRGB lineare\n(AHD + WB + cam→sRGB, gamma preview)')
axes[0,1].axis('off')

for idx, (cmap, name) in enumerate([('Reds','R'),('Greens','G'),('Blues','B')]):
    axes[0, 2+idx].imshow(np.clip(thumb_lin[:,:,idx], 0, 1) ** (1/2.2),
                           cmap=cmap, interpolation='lanczos')
    axes[0, 2+idx].set_title(f'Canale {name} — lineare\n(gamma preview)')
    axes[0, 2+idx].axis('off')

# ── Riga 1: istogrammi + luminanza ────────────────────────────────────────────
for idx, (col, name) in enumerate(zip(['#ef5350','#66bb6a','#42a5f5'], 'RGB')):
    ch = thumb_lin[:,:,idx].flatten()
    axes[1,idx].hist(ch[ch > 5e-4], bins=200, color=col, density=True, alpha=0.85, log=True)
    axes[1,idx].set_title(f'Istogramma canale {name}\n(scala log — linearità visibile)')
    axes[1,idx].set_xlabel('$I^{{lin}}$')
    axes[1,idx].set_ylabel('densità')

im = axes[1,3].imshow(lum[::TH,::TH] ** (1/2.2), cmap='gray', interpolation='lanczos')
axes[1,3].set_title('Luminanza CIE Y\n$0.2126R + 0.7152G + 0.0722B$')
axes[1,3].axis('off')
plt.colorbar(im, ax=axes[1,3], fraction=0.046)

axes[1,4].axis('off')
info = (
    'rawpy.postprocess() — parametri\n'
    '─────────────────────────────────\n'
    '  demosaic:    AHD          §6.2.3\n'
    '  output_color: sRGB        §6.2.6-7\n'
    '  output_bps:  16 bit\n'
    '  use_camera_wb: True       §6.2.5\n'
    '  no_auto_bright: True\n'
    '  gamma: (1,1)  ← lineare!\n'
    '  highlight: Clip\n'
    '─────────────────────────────────\n'
    f'  Output: {I_lin.shape}\n'
    f'  Range:  [{I_lin.min():.3f}, {I_lin.max():.3f}]'
)
axes[1,4].text(0.5, 0.5, info, transform=axes[1,4].transAxes,
               fontsize=8.5, va='center', ha='center',
               bbox=dict(boxstyle='round', facecolor='#1a3a1a', alpha=0.85),
               family='monospace', color='#a5d6a7')

fig.suptitle('Step 1 — rawpy.postprocess(): AHD + WB + cam→XYZ→sRGB lineare  [Sez. 6.2.3–6.2.7]',
             fontsize=13)
plt.tight_layout()
plt.show()

---
## Step 2 — Gamma encoding: sRGB lineare → sRGB standard
### Sezione 6.2.8

I valori $I^{lin}$ sono **linearmente proporzionali all'irradianza fisica**.
Il sistema visivo umano ha però una risposta non lineare alla luce
(approssimativamente logaritmica), quindi i valori lineari devono essere
codificati con una funzione non lineare prima di essere visualizzati
o salvati come JPEG/TIFF 8-bit.

### Funzione di trasferimento sRGB — EOTF (IEC 61966-2-1)

$$I^{sRGB}_c = \gamma_{sRGB}(I^{lin}_c) = \begin{cases}
12.92 \cdot I^{lin}_c & I^{lin}_c \leq 0.0031308 \\
1.055 \cdot (I^{lin}_c)^{1/2.4} - 0.055 & I^{lin}_c > 0.0031308
\end{cases}$$

La curva è una **gamma ≈ 2.2 con zona lineare a bassa luminanza**:
- La zona lineare ($12.92 \cdot x$ per $x \leq 0.0031308$) evita il rumore di
  quantizzazione nelle ombre profonde, dove $x^{1/2.4}$ avrebbe derivata infinita
  in $x=0$
- La soglia $0.0031308$ garantisce continuità e derivata continua nel punto di giunzione
- I coefficienti $1.055$ e $0.055$ garantiscono la continuità della derivata

### Funzione inversa — OETF

Necessaria quando si caricano target JPEG per il calcolo della loss in spazio lineare:

$$I^{lin}_c = \gamma_{sRGB}^{-1}(I^{sRGB}_c) = \begin{cases}
I^{sRGB}_c \,/\, 12.92 & I^{sRGB}_c \leq 0.04045 \\
\left(\dfrac{I^{sRGB}_c + 0.055}{1.055}\right)^{2.4} & I^{sRGB}_c > 0.04045
\end{cases}$$

Nota: la soglia inversa è $0.04045 = 12.92 \times 0.0031308$.

### Proprietà di questa implementazione

Questa è un'implementazione **manuale e vettorizzata** su numpy, differenziabile
rispetto all'input tranne nel punto di giunzione (dove però la derivata è continua).
La verifica del round-trip $\gamma^{-1}(\gamma(x)) = x$ deve dare errore $< 10^{-6}$.

In [ ]:
def gamma_srgb(x):
    """sRGB EOTF: lineare → standard (IEC 61966-2-1)."""
    x = np.clip(x, 0.0, 1.0)
    return np.where(x <= 0.0031308,
                    12.92 * x,
                    1.055 * x**(1.0/2.4) - 0.055)

def gamma_srgb_inv(x):
    """sRGB OETF: standard → lineare (inversione esatta)."""
    x = np.clip(x, 0.0, 1.0)
    return np.where(x <= 0.04045,
                    x / 12.92,
                    ((x + 0.055) / 1.055) ** 2.4)

I_srgb = gamma_srgb(I_lin)

# ── Verifica round-trip ───────────────────────────────────────────────────────
err_rt = np.abs(gamma_srgb_inv(I_srgb) - I_lin).max()
print(f'I_lin  — range [{I_lin.min():.4f}, {I_lin.max():.4f}]  mean={I_lin.mean():.4f}')
print(f'I_srgb — range [{I_srgb.min():.4f}, {I_srgb.max():.4f}]  mean={I_srgb.mean():.4f}')
print(f'Errore round-trip γ⁻¹(γ(x)):  {err_rt:.2e}  (deve essere < 1e-6)')

TH = 8
thumb_lin2 = I_lin[::TH, ::TH]
thumb_srgb = I_srgb[::TH, ::TH]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# ── Riga 0: immagini ──────────────────────────────────────────────────────────
axes[0,0].imshow(np.clip(thumb_lin2, 0, 1) ** (1/2.2), interpolation='lanczos')
axes[0,0].set_title('$I^{lin}$ — sRGB lineare\n(gamma preview per display)')
axes[0,0].axis('off')

axes[0,1].imshow(np.clip(thumb_srgb, 0, 1), interpolation='lanczos')
axes[0,1].set_title('$I^{sRGB}$ — sRGB standard\n(dopo gamma encoding — display-ready)')
axes[0,1].axis('off')

# differenza (sRGB - lineare), amplificata per rendere visibile la non-linearità
diff = np.clip((thumb_srgb - thumb_lin2) * 2.5 + 0.5, 0, 1)
axes[0,2].imshow(diff, interpolation='lanczos')
axes[0,2].set_title('$(I^{sRGB} - I^{lin}) \times 2.5 + 0.5$\n(midgray = zero, luci = gamma alzata le ombre)')
axes[0,2].axis('off')

# ── Riga 1: curve + istogrammi ────────────────────────────────────────────────
x_v = np.linspace(0, 1, 2000)
axes[1,0].plot(x_v, gamma_srgb(x_v),  color='#80cbc4', lw=2.5, label='$\gamma_{sRGB}$ (1/2.4 + zona lin.)')
axes[1,0].plot(x_v, x_v**(1/2.2),     color='#ffb74d', lw=1.5, ls='--', label='$x^{1/2.2}$ (gamma pura)')
axes[1,0].plot(x_v, x_v,              color='#555',    lw=1,   ls=':',  label='identità')
axes[1,0].axvline(0.0031308, color='white', lw=0.8, ls='--', alpha=0.5, label='soglia 0.003')
axes[1,0].set_title('EOTF sRGB vs gamma 2.2 pura')
axes[1,0].set_xlabel('$I^{lin}$')
axes[1,0].set_ylabel('$I^{sRGB}$')
axes[1,0].legend(fontsize=8)

# zoom zona lineare
x_z = np.linspace(0, 0.015, 1000)
axes[1,1].plot(x_z, gamma_srgb(x_z),  color='#80cbc4', lw=2.5, label='$\gamma_{sRGB}$')
axes[1,1].plot(x_z, x_z**(1/2.4),     color='#ef5350', lw=1.5, ls='--', label='$x^{1/2.4}$ (no zona lin.)')
axes[1,1].plot(x_z, 12.92 * x_z,      color='#ffb74d', lw=1.5, ls=':',  label='$12.92x$ (zona lin.)')
axes[1,1].axvline(0.0031308, color='yellow', lw=1.2, ls='--', label='soglia 0.0031308')
axes[1,1].set_title('Zoom zona lineare — ombre profonde')
axes[1,1].set_xlabel('$I^{lin}$')
axes[1,1].legend(fontsize=7)

axes[1,2].hist(thumb_lin2.flatten(),  bins=200, color='#4fc3f7', alpha=0.7,
               density=True, log=True, label='$I^{lin}$ (lineare)')
axes[1,2].hist(thumb_srgb.flatten(),  bins=200, color='#ce93d8', alpha=0.7,
               density=True, log=True, label='$I^{sRGB}$ (gamma)')
axes[1,2].set_title('Distribuzione lineare vs sRGB\n(la gamma redistribuisce verso le ombre)')
axes[1,2].set_xlabel('valore pixel')
axes[1,2].legend(fontsize=8)

fig.suptitle('Step 2 — Gamma encoding: $I^{lin}$ → $I^{sRGB}$  [Sez. 6.2.8, IEC 61966-2-1]',
             fontsize=13)
plt.tight_layout()
plt.show()

---
## Step 3 — Downscaling adattivo (resolution-agnostic)
### Sezione 6.2.9

Le immagini RAW moderne vanno da 24 MP (es. Nikon Z6: $6048\times 4024$) a
oltre 60 MP (es. Sony A7R V: $9504\times 6336$). Per il training su GPU è
necessario lavorare a dimensioni gestibili, preservando la **coerenza geometrica**
tra sorgente e target.

### Fattore di scala uniforme

Sia $(H_0, W_0)$ la dimensione dell'immagine sRGB dopo demosaicatura.
Il fattore di scala per portare il lato lungo a $L_{train}$ pixel è:

$$s = \frac{L_{train}}{\max(H_0, W_0)} \in (0, 1]$$

Le dimensioni ridotte con arrotondamento all'intero più vicino:

$$H_s = \lfloor s \cdot H_0 \rceil, \qquad W_s = \lfloor s \cdot W_0 \rceil$$

### Filtro Lanczos ($a = 3$) — anti-aliasing

Il campionamento con step $1/s > 1$ introduce **aliasing** (repliche spettrali
delle frequenze oltre Nyquist) se non preceduto da filtro passa-basso.
Il filtro di Lanczos è il prodotto di due seno-cardinali:

$$L(x) = \begin{cases}
\text{sinc}(x)\,\text{sinc}(x/3) & |x| < 3 \\
0 & |x| \geq 3
\end{cases}, \qquad \text{sinc}(x) = \frac{\sin(\pi x)}{\pi x}$$

La funzione di ricampionamento per il pixel output $(i_s, j_s)$:

$$I^{sRGB}_c(i_s, j_s) = \sum_{i'} \sum_{j'} I^{sRGB}_c(i', j') \cdot
L\!\left(\frac{i'}{s} - i_s\right) \cdot L\!\left(\frac{j'}{s} - j_s\right)$$

La somma coinvolge una finestra $6\times 6$ pixel attorno a ogni punto campionato.

### Proprietà resolution-agnostic

Il fattore $s$ è definito in termini delle dimensioni relative dell'immagine:
il **medesimo modello** addestrato su crop $768\times 512$ può elaborare
all'inferenza immagini a qualsiasi risoluzione senza modifiche architetturali.

### Coerenza della coppia $(I^{src}_s, I^{tgt}_s)$

Il target JPEG deve essere ricampionato con lo **stesso identico** $s$,
le stesse dimensioni $(H_s, W_s)$ e lo stesso filtro Lanczos, per preservare
la corrispondenza spaziale pixel-to-pixel. Qualsiasi disallineamento sub-pixel
invaliderebbe la loss pixel-wise.

> ⚠️ **Nota implementativa:** PIL/Pillow applica il filtro Lanczos come
> approssimazione discreta con $a=3$ (implementazione C ottimizzata).
> Il downscaling avviene **in spazio sRGB gamma-encoded** (su $I^{sRGB}$),
> non in spazio lineare. Questo è il comportamento standard nei workflow
> fotografici professionali — fare il resize in spazio lineare produrrebbe
> aloni chiari attorno ai bordi scuri (il noto "halo problem").

In [ ]:
H0, W0 = I_srgb.shape[:2]
s   = TRAIN_LONG_SIDE / max(H0, W0)
H_s = round(s * H0)
W_s = round(s * W0)

print(f'Risoluzione originale:  {H0} × {W0}  ({H0*W0/1e6:.1f} MP)')
print(f'Fattore di scala s:     {s:.6f}')
print(f'Risoluzione target:     {H_s} × {W_s}  (L_train={TRAIN_LONG_SIDE})')
print(f'Compressione area:      {(H0*W0)/(H_s*W_s):.1f}×  '
      f'({H0*W0/1e6:.1f} MP → {H_s*W_s/1e6:.2f} MP)')

# ── Downsampling Lanczos (PIL, implementazione C ottimizzata, a=3) ─────────────
img_pil = PILImage.fromarray((I_srgb * 255).astype(np.uint8))
img_dn  = img_pil.resize((W_s, H_s), PILImage.LANCZOS)
X = np.array(img_dn).astype(np.float32) / 255.0

print(f'Output X: {X.shape}  [{X.min():.3f}, {X.max():.3f}]')

# ── Kernel Lanczos analitico (per visualizzazione) ────────────────────────────
def lanczos_kernel(x, a=3):
    x = np.abs(x)
    return np.where(x == 0, 1.0,
           np.where(x < a, np.sinc(x) * np.sinc(x / a), 0.0))

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

TH = 8
axes[0,0].imshow(I_srgb[::TH, ::TH], interpolation='lanczos')
axes[0,0].set_title(f'$I^{{sRGB}}$ originale {H0}×{W0}\n(thumbnail 1:{TH})')
axes[0,0].axis('off')

axes[0,1].imshow(X, interpolation='lanczos')
axes[0,1].set_title(f'$X$ downscaled {H_s}×{W_s}\n(s = {s:.4f}, filtro Lanczos $a=3$)')
axes[0,1].axis('off')

# crop centrale pixel-accurate
ch = round(100 * s); cw = round(140 * s)
crop_ds = X[H_s//2 - ch : H_s//2 + ch, W_s//2 - cw : W_s//2 + cw]
axes[0,2].imshow(crop_ds, interpolation='nearest')
axes[0,2].set_title('Crop centrale — display nearest\n(per verificare nitidezza Lanczos)')
axes[0,2].axis('off')

# Kernel Lanczos
x_k = np.linspace(-3.5, 3.5, 2000)
y_k = lanczos_kernel(x_k)
axes[1,0].plot(x_k, y_k, color='#80cbc4', lw=2.5)
axes[1,0].fill_between(x_k, y_k, alpha=0.15, color='#80cbc4')
axes[1,0].axhline(0, color='#555', lw=0.8)
for i in range(-3, 4):
    axes[1,0].axvline(i, color='#333', lw=0.5, ls='--')
axes[1,0].scatter([0], [1], color='#ffb74d', zorder=5, s=40)
axes[1,0].set_title('Kernel Lanczos $L(x)$ con $a=3$\n$\mathrm{sinc}(x)\cdot\mathrm{sinc}(x/3)$')
axes[1,0].set_xlabel('$x$ (distanza dal punto campionato)')
axes[1,0].set_ylabel('$L(x)$')

# PSD confronto — analisi frequenziale
def compute_psd(gray):
    f = scipy_fft.fft2(gray)
    return np.log1p(np.abs(scipy_fft.fftshift(f))**2)

gray_orig = I_srgb[::TH, ::TH, 1]   # canale G a piena frequenza (thumbnail)
gray_ds   = X[:, :, 1]               # canale G downscaled

axes[1,1].imshow(compute_psd(gray_orig), cmap='inferno', interpolation='lanczos')
axes[1,1].set_title('PSD canale G — originale\n(spazio frequenze, scala log)')
axes[1,1].axis('off')

axes[1,2].imshow(compute_psd(gray_ds), cmap='inferno', interpolation='lanczos')
axes[1,2].set_title('PSD canale G — downscaled\n(freq. oltre Nyquist rimosse da Lanczos)')
axes[1,2].axis('off')

fig.suptitle(f'Step 3 — Downscaling Lanczos: {H0}×{W0} → {H_s}×{W_s}  [Sez. 6.2.9]',
             fontsize=13)
plt.tight_layout()
plt.show()

---
## Step 4 — Normalizzazione ImageNet → tensore di input
### Sezione 6.2.10

L'output dell'intera pipeline pre-neural è il tensore:
$$\mathbf{X} = I^{sRGB}_s \in [0,1]^{H_s \times W_s \times 3}$$

Prima di entrare nel modello, viene normalizzato con le **statistiche del
dataset ImageNet** per compatibilità con i pesi pre-addestrati di EfficientNet-B4:

$$\hat{X}_c(i,j) = \frac{X_c(i,j) - \mu_c^{IN}}{\sigma_c^{IN}}$$

con:
$$\boldsymbol{\mu}^{IN} = (0.485,\ 0.456,\ 0.406), \qquad
\boldsymbol{\sigma}^{IN} = (0.229,\ 0.224,\ 0.225)$$

### Motivazione della normalizzazione ImageNet

EfficientNet-B4 è stato pre-addestrato su ImageNet con input in sRGB standard
normalizzato con queste statistiche. Usare le stesse statistiche:

1. **Preserva la distribuzione attesa** dai layer BatchNorm del backbone
   (i layer BN hanno parametri calibrati per questa distribuzione)
2. **Porta** $\hat{X} \approx \mathcal{N}(0, \mathbf{I})$ — condizione favorevole
   per la stabilità del gradiente durante il fine-tuning
3. **Non richiede ri-addestramento** del backbone da zero

### Tensore finale in formato PyTorch

$$\hat{\mathbf{X}} \in \mathbb{R}^{B \times 3 \times H_s \times W_s}$$

Il formato PyTorch mette i canali **prima** delle dimensioni spaziali
(channel-first, NCHW), al contrario di numpy e PIL che usano NHWC.

In [ ]:
mu_IN    = np.array([0.485, 0.456, 0.406], dtype=np.float32)
sigma_IN = np.array([0.229, 0.224, 0.225], dtype=np.float32)

X_hat    = (X - mu_IN) / sigma_IN
X_tensor = X_hat.transpose(2, 0, 1)[np.newaxis]   # NHWC → NCHW: (1, 3, H_s, W_s)

print(f'X      ∈ [0,1]:   shape={X.shape}         range=[{X.min():.3f}, {X.max():.3f}]')
print(f'X̂      norm.:    shape={X_hat.shape}  range=[{X_hat.min():.3f}, {X_hat.max():.3f}]')
print(f'X_tensor PyTorch: shape={X_tensor.shape}')
print()
for i, name in enumerate('RGB'):
    ch = X_hat[:,:,i]
    print(f'  Canale {name}: mean={ch.mean():+.4f}  std={ch.std():.4f}  '
          f'(target ~0, ~1)   μ={mu_IN[i]:.3f}  σ={sigma_IN[i]:.3f}')

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

# ── Riga 0: X originale + canali normalizzati ─────────────────────────────────
axes[0,0].imshow(X, interpolation='lanczos')
axes[0,0].set_title(f'$X \in [0,1]^3$\n{X.shape[0]}×{X.shape[1]} px (input normalizzazione)')
axes[0,0].axis('off')

vabs = max(abs(X_hat.min()), abs(X_hat.max()))
for idx, name in enumerate('RGB'):
    im = axes[0, 1+idx].imshow(X_hat[:,:,idx], cmap='RdBu_r',
                                vmin=-vabs, vmax=vabs, interpolation='lanczos')
    axes[0, 1+idx].set_title(f'$\hat{{X}}$ canale {name}\n'
                              f'$(X_{{{name}}} - {mu_IN[idx]:.3f}) / {sigma_IN[idx]:.3f}$')
    axes[0, 1+idx].axis('off')
    plt.colorbar(im, ax=axes[0, 1+idx], fraction=0.046)

# ── Riga 1: istogrammi + riquadro info ────────────────────────────────────────
for idx, (name, col) in enumerate(zip('RGB', ['#ef5350','#66bb6a','#42a5f5'])):
    ch = X_hat[:,:,idx].flatten()
    axes[1,idx].hist(ch, bins=120, color=col, density=True, alpha=0.85)
    x_g = np.linspace(ch.min(), ch.max(), 400)
    axes[1,idx].plot(x_g, sp_norm.pdf(x_g, 0, 1), 'w--', lw=1.8, label='$\mathcal{N}(0,1)$')
    axes[1,idx].set_title(f'$\hat{{X}}[{name}]$  '
                           f'mean={ch.mean():+.3f}  std={ch.std():.3f}')
    axes[1,idx].set_xlabel('valore normalizzato')
    axes[1,idx].legend(fontsize=8)

axes[1,3].axis('off')
info = (
    'Tensore finale (formato PyTorch)\n'
    '─────────────────────────────────\n'
    f'  shape: (B, C, H, W)\n'
    f'         (1, 3, {X_hat.shape[0]}, {X_hat.shape[1]})\n\n'
    f'  layout: NCHW (channel-first)\n'
    f'  dtype:  float32\n'
    f'  range:  [{X_hat.min():.2f}, {X_hat.max():.2f}]\n\n'
    f'  μ_IN = (0.485, 0.456, 0.406)\n'
    f'  σ_IN = (0.229, 0.224, 0.225)\n\n'
    f'  → CNN stem EfficientNet-B4'
)
axes[1,3].text(0.5, 0.5, info, transform=axes[1,3].transAxes,
               fontsize=9, va='center', ha='center',
               bbox=dict(boxstyle='round', facecolor='#1a2a3a', alpha=0.9),
               family='monospace', color='#b2ebf2')

fig.suptitle('Step 4 — Normalizzazione ImageNet → tensore $\hat{\mathbf{X}}$  [Sez. 6.2.10]',
             fontsize=13)
plt.tight_layout()
plt.show()

---
## Riepilogo — Pipeline completa e Teorema di Invarianza
### Sezione 6.2.11

La pipeline completa (Sezione 6.2.11):

$$\mathbf{R}
\xrightarrow{\substack{\text{6.2.1} \\ \text{linearize}}} R_{norm}
\xrightarrow{\substack{\text{6.2.3–7} \\ \texttt{postprocess()}}} \mathbf{I}^{lin}
\xrightarrow{\substack{\text{6.2.8} \\ \gamma_{sRGB}}} \mathbf{I}^{sRGB}
\xrightarrow{\substack{\text{6.2.9} \\ \text{Lanczos}}} \mathbf{X}
\xrightarrow{\substack{\text{6.2.10} \\ \mu^{IN}/\sigma^{IN}}} \hat{\mathbf{X}}$$

### Teorema 6 — Invarianza alla Risoluzione

**Enunciato.** Sia $f_\theta$ il modello HybridStyleNet. Per qualsiasi coppia di
risoluzioni $(H_1, W_1)$ e $(H_2, W_2)$ con stesso aspect ratio $H_1/W_1 = H_2/W_2$:

$$f_\theta\bigl(\text{Resize}_{H_2\to H_1}(\hat{\mathbf{X}}_{H_2\times W_2})\bigr)
\approx f_\theta\bigl(\hat{\mathbf{X}}_{H_1\times W_1}\bigr)$$

a meno di errori di ricampionamento $O(1/\min(H_1,H_2))$.

**Argomento (4 componenti):**

(a) **Convoluzioni** (EfficientNet-B4): kernel locali, risposta dipende
solo dai pattern locali — indipendente dalla risoluzione assoluta.

(b) **Swin Transformer con RoPE**: codifica le distanze *relative* tra token,
non le posizioni assolute. Cambiare la risoluzione cambia $T = H_s W_s / 32^2$
ma non la semantica delle rappresentazioni.

(c) **Bilateral grid**: parametrizzata in coordinate normalizzate
$x_g(j) = \frac{j}{W-1}(W_g-1)$ — la stessa trasformazione cromatica è applicata
alla stessa posizione relativa a qualsiasi risoluzione.

(d) **Confidence mask + blending finale**: operazioni pixel-wise e bilineare,
invarianti per costruzione. $\square$

In [ ]:
# ── Griglia comparativa — tutti gli step della pipeline ──────────────────────
TH = 8
stages = [
    (thumb_raw ** (1/2.2),
     'Step 0\nRAW grezzo $R_{norm}$\n(mosaico CFA, Sez. 6.2.1)'),
    (np.clip(I_lin[::TH,::TH], 0, 1) ** (1/2.2),
     'Step 1\n$I^{lin}$ — sRGB lineare\n(AHD+WB+cam→sRGB, Sez. 6.2.3–7)'),
    (np.clip(I_srgb[::TH,::TH], 0, 1),
     'Step 2\n$I^{sRGB}$ — gamma encoded\n(Sez. 6.2.8, IEC 61966-2-1)'),
    (X,
     f'Step 3\n$X$ downscaled {H_s}×{W_s}\n(Lanczos $s$={s:.3f}, Sez. 6.2.9)'),
    (np.clip((X_hat - X_hat.min()) / (X_hat.max() - X_hat.min()), 0, 1),
     'Step 4\n$\hat{X}$ normalizzato\n(ImageNet $\mu/\sigma$, Sez. 6.2.10)'),
]

fig, axes = plt.subplots(1, 5, figsize=(24, 5.5))
for ax, (img, title) in zip(axes, stages):
    ax.imshow(np.clip(img, 0, 1), cmap='gray' if img.ndim == 2 else None,
              interpolation='lanczos')
    ax.set_title(title, fontsize=9.5, pad=6)
    ax.axis('off')

fig.suptitle(
    f'Pipeline completa  —  {os.path.basename(RAW_PATH)}\n'
    f'Input: {H_raw}×{W_raw} px  ({H_raw*W_raw/1e6:.1f} MP)  →  '
    f'Output tensore: {X_tensor.shape}  float32',
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.show()

print('\n✓ Pipeline completata.')
print(f'  Input RAW:      {H_raw} × {W_raw} px  ({H_raw*W_raw/1e6:.1f} MP)')
print(f'  Output tensore: {X_tensor.shape}  dtype={X_tensor.dtype}')
print(f'  Compressione:   {H_raw*W_raw / (H_s*W_s):.1f}× (area)')

---
## Funzione `raw_to_tensor()` — implementazione production-ready

Racchiude l'intera pipeline in una singola funzione riutilizzabile.
Compatibile con **ARW, DNG, NEF, CR3, CR2, RAF, RW2, ORF, PEF** e qualsiasi
formato supportato da libraw/rawpy.

In [ ]:
def raw_to_tensor(
    raw_path      : str,
    long_side     : int  = 768,
    imagenet_norm : bool = True,
    return_steps  : bool = False,
) -> np.ndarray:
    """
    Pipeline completa RAW → tensore normalizzato (Sezione 6.2).

    Parametri
    ---------
    raw_path      : percorso file RAW (ARW, DNG, NEF, CR3, RAF, ...)
    long_side     : L_train — dimensione lato lungo per downscaling
    imagenet_norm : se True, normalizza con μ/σ ImageNet (Sez. 6.2.10)
    return_steps  : se True, ritorna anche dizionario degli step intermedi

    Ritorna
    -------
    np.ndarray  shape (1, 3, H_s, W_s)  float32  — tensore PyTorch NCHW
    """
    import rawpy
    import numpy as np
    from PIL import Image as PILImage

    steps = {}

    # ── Sezioni 6.2.3–6.2.7: demosaic + WB + color matrix (rawpy) ────────────
    raw = rawpy.imread(raw_path)
    rgb16 = raw.postprocess(
        demosaic_algorithm = rawpy.DemosaicAlgorithm.AHD,
        output_color       = rawpy.ColorSpace.sRGB,
        output_bps         = 16,
        use_camera_wb      = True,
        no_auto_bright     = True,
        gamma              = (1, 1),          # output lineare puro
        highlight_mode     = rawpy.HighlightMode.Clip,
    )
    I_lin = rgb16.astype(np.float32) / 65535.0
    if return_steps: steps['I_lin'] = I_lin

    # ── Sezione 6.2.8: gamma encoding sRGB ───────────────────────────────────
    I_srgb = np.where(I_lin <= 0.0031308,
                      12.92 * I_lin,
                      1.055 * I_lin**(1.0/2.4) - 0.055)
    if return_steps: steps['I_srgb'] = I_srgb

    # ── Sezione 6.2.9: downscaling Lanczos ───────────────────────────────────
    H0, W0 = I_srgb.shape[:2]
    s   = long_side / max(H0, W0)
    H_s = round(s * H0)
    W_s = round(s * W0)
    X = np.array(
        PILImage.fromarray((I_srgb * 255).astype(np.uint8))
                .resize((W_s, H_s), PILImage.LANCZOS)
    ).astype(np.float32) / 255.0
    if return_steps: steps['X'] = X

    # ── Sezione 6.2.10: normalizzazione ImageNet ──────────────────────────────
    if imagenet_norm:
        mu    = np.array([0.485, 0.456, 0.406], np.float32)
        sigma = np.array([0.229, 0.224, 0.225], np.float32)
        X = (X - mu) / sigma

    # NHWC → NCHW (formato PyTorch)
    tensor = X.transpose(2, 0, 1)[np.newaxis]

    if return_steps:
        steps['tensor'] = tensor
        return tensor, steps
    return tensor


# ── Test ──────────────────────────────────────────────────────────────────────
print('Test raw_to_tensor() ...')
t, steps_out = raw_to_tensor(RAW_PATH, return_steps=True)
print(f'✓  Output:          {t.shape}  dtype={t.dtype}')
print(f'   range:           [{t.min():.3f}, {t.max():.3f}]')
print(f'   Step intermedi:  {list(steps_out.keys())}')